# 4.1 CNN Image Classification

使用 Fashion-MNIST 從零建立 CNN 影像分類模型。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report

## 1. 載入資料

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print(x_train.shape, x_test.shape)

## 2. 觀察圖片

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap='gray')
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()

## 3. 建立 CNN

In [ ]:
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 4. 訓練模型

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=8,
    batch_size=128,
    verbose=1
)

pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='Accuracy Curve')
plt.show()

## 5. 評估模型

In [ ]:
train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

pd.DataFrame({'dataset':['train','test'], 'loss':[train_loss,test_loss], 'accuracy':[train_acc,test_acc]})

In [ ]:
y_prob = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=class_names))

## 6. 預測新圖片

In [ ]:
idx = 0
prob = model.predict(x_test[idx:idx+1], verbose=0)[0]
print('pred:', class_names[np.argmax(prob)], 'true:', class_names[y_test[idx]], 'confidence:', np.max(prob))
plt.imshow(x_test[idx].squeeze(), cmap='gray')
plt.axis('off')
plt.show()

## 7. 如何套用自己的資料

若要改成自己的圖片資料，通常會改用 `image_dataset_from_directory` 載入資料夾。圖片大小會影響輸入 shape，灰階或彩色圖片會影響 channel 數，類別數則會影響最後一層神經元數量。


## 8. 小結

CNN 影像分類的核心流程是整理圖片 shape、建立卷積與池化層、用 softmax 輸出類別機率，最後用 confusion matrix 與錯誤案例檢查模型表現。
